#  Week 3, Day 4 — June 4, 2026
## Compile the Final Integrated Master Table



In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import os, json, warnings
warnings.filterwarnings('ignore')
BASE_DIR = "/content/drive/MyDrive/Ocean_Plastic_Project"
DIRS = {"raw": BASE_DIR+"/data/raw","processed": BASE_DIR+"/data/processed",
        "metadata": BASE_DIR+"/data/metadata","visualizations": BASE_DIR+"/visualizations"}
LAT_MIN,LAT_MAX,LON_MIN,LON_MAX = 5,30,65,95
YEAR_MIN,YEAR_MAX = 2015,2026

In [ ]:
plastic_grid = pd.read_csv(DIRS['processed']+'/plastic_grid_aggregated.csv')
species_grid = pd.read_csv(DIRS['processed']+'/species_grid_aggregated.csv')
sst_monthly  = pd.read_csv(DIRS['processed']+'/sst_monthly_aggregated.csv')
df_world     = pd.read_csv(DIRS['processed']+'/plastic_world.csv')
print(f'plastic_grid: {plastic_grid.shape}, species_grid: {species_grid.shape}, sst: {sst_monthly.shape}')

## Step 1: Outer Join Plastic + Species on Grid + Year

In [ ]:
# OUTER join: keep all grid cells even if only one dataset has data
master = pd.merge(
    plastic_grid, species_grid,
    on=['grid_lat','grid_lon','year'],
    how='outer'
)

print(f'After plastic + species join: {len(master):,} rows × {master.shape[1]} cols')
print(f'Nulls introduced: {master.isnull().sum().sum()}')
print()
print('Null breakdown:')
print(master.isnull().sum()[master.isnull().sum()>0])

## Step 2: Add SST on Year + Month

In [ ]:
master = pd.merge(master, sst_monthly, on=['year','month'], how='left')
print(f'After adding SST: {len(master):,} rows × {master.shape[1]} cols')

## Step 3: Add Country + Waste Source from World Dataset

In [ ]:
# All our records are in India's maritime zone
master['Country'] = 'India'

# Bring in India's waste characteristics
india = df_world[df_world['Country']=='India'].iloc[0]
master['Waste_Source']        = india['Main_Sources']       # 'Consumer_Goods'
master['Coastal_Risk']        = india['Coastal_Waste_Risk'] # 'High'
master['Recycling_Rate_Pct']  = india['Recycling_Rate']     # 11.5%

print(f'Added Country, Waste_Source, Coastal_Risk, Recycling_Rate_Pct')
print()
print('Master Table schema:')
print(master.dtypes)

## Step 4: Assign Ocean Zone (if not already present)

In [ ]:
def assign_zone(lat, lon):
    if   5 <= lat <= 25 and 65 <= lon <= 75: return 'Arabian_Sea'
    elif 5 <= lat <= 22 and 75 <= lon <= 90: return 'Bay_of_Bengal'
    elif 10<= lat <= 25 and 90 <= lon <= 95: return 'Andaman_Sea'
    elif 5 <= lat <= 12 and 72 <= lon <= 80: return 'Lakshadweep_Gulf_of_Mannar'
    elif 20<= lat <= 30 and 65 <= lon <= 75: return 'Gulf_of_Kutch'
    else: return 'Indian_Ocean'

master['Ocean_Zone'] = master.apply(lambda r: assign_zone(r['grid_lat'],r['grid_lon']), axis=1)

print('Ocean Zone distribution in Master Table:')
print(master['Ocean_Zone'].value_counts())
print()
print('=== MASTER TABLE PREVIEW ===')
print(master.head(5).to_string())
print()
print(f'Final shape: {master.shape[0]:,} rows × {master.shape[1]} cols')

In [ ]:
master.to_csv(DIRS['processed']+'/master_table_v1.csv', index=False)
print(f'Master Table saved: master_table_v1.csv ({master.shape[0]:,} rows × {master.shape[1]} cols) ✅')
print()
print('Required attributes check:')
required = ['Ocean_Zone','Country','year','month','Plastic_Density_kg_km2',
            'Waste_Source','SST_C','Species_Count','Taxonomic_Groups']
for col in required:
    present = col in master.columns
    print(f'  {col:<30} {"✅" if present else "❌ MISSING"}')

## ✅ Day 4 Week 3 Complete — June 4, 2026

**Master Table:** 1,228 rows covering the North Indian Ocean 2015 period.

**Next: June 5 — Join Integrity Checks**